# 2022 -Geodaten für Updates der Quartierspiegel aufbereiten
Datum: 28.02.2022

Mit Colab kann **dieses Jupyter-Notebook interaktiv im Browser** gestartet werden:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DonGoginho/myPy/blob/main/quartierspiegel/quartierspiegel_geodaten_aufbereiten.ipynb)

## Lebensqualitätskarten in den [Quartierspiegeln](https://www.stadt-zuerich.ch/prd/de/index/statistik/publikationen-angebote/publikationen/Quartierspiegel.html)

Für diese Karte werden diverse POIs ausgelesen und als Punkte dargestellt. Damit die Statistik der POIs im Bericht ausgewertet werden kann, hat Stefanie ein R-Skript erstellt, mit dem sie Auswertungen zu den POIs macht. Dazu braucht sie Shapefiles, die vom Geoserver exportiert und historisiert abgespeichert werden.

Bisher habe ich dazu ein Skipt verwendet. Damit dieser Prozess noch transparenter und einfacher wird, erstelle ich nun dieses Notebook.

Die QGIS-Projekte für die Quartierspiegel sind unter 
- `O:\GIS\Sandbox\*JAHR*\quartierspiegel\minimaps` abgelegt.

Dieses Notebook hilft die in der Karte aktiven Layer – die für die Karte vom [**Geoserver**](https://www.stadt-zuerich.ch/geodaten/) als View bezogen sind – als shp zu exportieren.

<!---  
Dieses Notebook verwendet als Vorlage die Webseite «[Kitas in Berlin - Jupyter Notebook](https://juanitorduz.github.io/kitas_berlin/)» 
---!>

## Importiere notwendige Packages:

In [1]:
#pip install geopandas altair fiona requests folium mplleaflet contextily seaborn datetime

In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
import altair as alt
import fiona
import requests
import mplleaflet
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from datetime import datetime


## Datenaufbereitung

### Globale Variabeln

Als Zeitvariable berechnen wir erst mal das heutige Datum. Wir können diese Werte als Dateinamen verwenden.

**Steuerungsvariable activate_date**: 
- False: in den Filenamen kommen beim Export KEINE Datenangaben mit
- True: die Exportfiles haben *FILENAMEN_DATUM* im Namen

In [3]:
activate_date = False;


#https://www.programiz.com/python-programming/datetime/strftime
now = datetime.now() # current date and time
year_today = now.strftime("%Y")

if activate_date == True:
    date_today = "_"+now.strftime("%Y-%m-%d")
    #store_o_gis = False
    
else:
    date_today =""
    #store_o_gis = True


In [4]:
print("Datum: "+date_today +" Jahr: "+year_today)

Datum:  Jahr: 2022


### Pfade


In [5]:
gis_ablagepfad = "//szh/ssz/data/GIS/Daten/Vektor/"+year_today+"/quartierspiegel/"
print(gis_ablagepfad )


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/


### Basis Geodaten

Für grundsätzliche räumliche Prüfungen importieren wir hier auch die 
- **Statistischen Quartiere**
- **Stadtgrenze** (aus der SSZ-Geodatenablage)



In [32]:
# Read abt. Geopandas https://geopandas.org/docs/user_guide/io.html
url0 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Statistische_Quartiere?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=adm_statistische_quartiere_map"
gdf_adm_statistische_quartiere_map = gpd.read_file(url0)
print('done')

done


In [33]:
# Read abt. Geopandas https://geopandas.org/docs/user_guide/io.html
pfad_grenze = r"\\szh\ssz\data\GIS\Daten\Vektor\_aktuell\adm_stadtgrenze_a.shp"
gdf_adm_stadtgrenze_lv95 = gpd.read_file(pfad_grenze)

gdf_adm_stadtgrenze_wgs84_all =gdf_adm_stadtgrenze_lv95.to_crs(epsg=4326)

print('done')


done


In [34]:
# Rename columns bei Bedarf
rename_cols = {
    'name': 'bfs_name',
}

columns_to_drop = ['objid']

gdf_adm_stadtgrenze_wgs84 = gdf_adm_stadtgrenze_wgs84_all \
    .copy() \
    .drop(columns_to_drop, axis=1) \
    .rename(columns=rename_cols) \
    

In [35]:
#gdf_adm_stadtgrenze_wgs84

## Geodaten für die Karte aufbereiten

Beziehe die notwendigen Geodaten vom Geoportal und exportiere sie als Shapefiles.


### Alterszentren:

Stefanie muss jedes Jahr mit der DA Alterszentren abklären, welche Alterszentren **ausgeschlossen** werden müssen, da nicht immer alle Bewohnt sind, wegen Umbau oder Ähnlichem. 
Aussderdem gibt es noch solche, die ausserhalb der Stadt liegen. Diese interessieren für die Quartierspiegel nicht und werden ebenfalls herausgefiltert.

#### Step 1: Daten vom Geoportal beziehen und prüfen.

In [36]:
url1 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Alterszentrum?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_alterszentrum_view"

gdf_poi_alterszentrum_view_imp_all = gpd.read_file(url1)

#gdf_poi_alterszentrum_view_imp_all.head(1)
#gdf_poi_alterszentrum_view_imp_all[['name','adresse', 'plz','geometry']]
#gdf_poi_alterszentrum_view_imp_all.info()

##### Einige Pflegezentren gehören offenbar **seit 2022** auch dazu... IN ABKLÄRUNG

In [37]:
url_pflegezentren="https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Pflegezentrum?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_pflegezentrum_view"
gdf_poi_pflegezentrum_view = gpd.read_file(url_pflegezentren)

In [38]:
#gdf_poi_pflegezentrum_view[['id', 'name','adresse', 'plz','ort','geometry']]
#gdf_poi_pflegezentrum_view.dtypes

Füge ein neues Attribut dazu, welches das erste Wort des Namens nimmt. Z.B. Gesundheitszentrum oder Pflegewohngruppe

In [46]:
gdf_poi_pflegezentrum_view[['typen']] = gdf_poi_pflegezentrum_view.name.str.split('\s+').str[0]
gdf_poi_pflegezentrum_view[['id','typen','name','adresse', 'plz','ort','geometry']].sort_values('id', ascending =True).head()

,id,typen,name,adresse,plz,ort,geometry
14,poi_pflegezentrum_view.10770,Start,Start up,Wiesenstrasse 10a,8952,Schlieren,POINT (8.44377 47.40007)
15,poi_pflegezentrum_view.10808,Hotel,Hotel Spirgarten,Lindenplatz 5,8048,Zürich,POINT (8.48554 47.38743)
16,poi_pflegezentrum_view.10958,Online,"Online Veranstaltung, Gesundheitszentren für d...",Eggbühlstrasse 23,8050,Zürich,POINT (8.54250 47.41615)
17,poi_pflegezentrum_view.11182,Gesundheitszentrum,Gesundheitszentrum für das Alter Bombach,Limmattalstrasse 371,8049,Zürich,POINT (8.48361 47.40422)
18,poi_pflegezentrum_view.11183,Gesundheitszentrum,Gesundheitszentrum für das Alter Entlisberg,Paradiesstrasse 45,8038,Zürich,POINT (8.52680 47.33460)


#### Step 2: Filtere diejenigen heraus, die nicht benötigt werden

Spatial Join, damit nur jene **innerhalb des Stadtgebiets** vorkommen. Hier sind zwei Möglichkeiten dargestellt, Verschnitt mit Stadtgrenze oder mit den Stadtquartieren. Wichti g ist, dass dabei  auch gerade die Attribute der Geodaten angehängt werden.

In [47]:
gdf_poi_alterszentrum_view_filter_stadtgebiet = gpd.sjoin(gdf_poi_alterszentrum_view_imp_all, gdf_adm_stadtgrenze_wgs84, how='inner', op='within')
#gdf_poi_alterszentrum_view_filter_stadtgebiet = gpd.sjoin(gdf_poi_alterszentrum_view_imp_all, gdf_adm_statistische_quartiere_map, how='inner', op='within')
#gdf_poi_alterszentrum_view_filter_stadtgebiet.info()



In [48]:
#Das selbe für die Pflegezentren 2022
gdf_poi_pflegezentrum_view_filter_stadtgebiet = gpd.sjoin(gdf_poi_pflegezentrum_view, gdf_adm_stadtgrenze_wgs84, how='inner', op='within')

In [49]:
#Temportärer Output zur Abstimmung mit dem GUD welche wir ausschliessen sollen und welche nicht...
gdf_poi_alterszentrum_view_filter_stadtgebiet[['name','adresse', 'plz','ort','geometry']].sort_values('name', ascending =True).to_excel("./output/listen/aktuelle_alterszentren_stadt"+date_today+".xlsx", index=False)
#gdf_poi_alterszentrum_view_filter_stadtgebiet[['name','adresse', 'plz','ort','geometry']].sort_values('name', ascending =True)

gdf_poi_pflegezentrum_view_filter_stadtgebiet[['id', 'typen', 'name','adresse', 'plz','ort','geometry']].sort_values('name', ascending =True).to_excel("./output/listen/aktuelle_pflegezentren_stadt"+date_today+".xlsx", index=False)
#gdf_poi_pflegezentrum_view_filter_stadtgebiet[['name','adresse', 'plz','ort','geometry']].sort_values('name', ascending =True)

Filtere explizt **vom GUD erwähnte Alterszentren** heraus. (Diejenigen, die nicht im Stadtgebiet liegen sind oben räumlich bereits herausgefiltert worden.)

Am 18.10.2021 wurden ausserdem vom GUD folgende gemeldet, die ausgeschlossen werden sollen:
- Alterszentrum Hardau
- Gästeappartement Alterszentrum Mittelleimbach

Am 28.02.2022 wurden folgende zum Ausschluss gemeldet:
- Gesundheitszentrum für das Alter Eichrain
- Online Veranstaltung
- Temporäres Gesundheitszentrum für das Alter Triemli


In [53]:
#2021:gdf_poi_alterszentrum_view_filtered = gdf_poi_alterszentrum_view_filter_stadtgebiet.query("name !='Alterszentrum Wolfswinkel'  & name !='Alterszentrum Buttenau' & name !='Alterszentrum Sonnenhof' & name !='Gästeappartement Alterszentrum Mittelleimbach'  & name !='Online Veranstaltung' & name !='Alterszentrum Hardau' & name !='Gästeappartement Mittelleimbach'") 
gdf_poi_alterszentrum_view_filtered = gdf_poi_alterszentrum_view_filter_stadtgebiet.query("name !='Gesundheitszentrum für das Alter Eichrain' & name !='Online Veranstaltung' & name !='Temporäres Gesundheitszentrum für das Alter Triemli'") 

#gdf_poi_alterszentrum_view_filtered.info()
#gdf_poi_alterszentrum_view_filtered

Filtere explizt **vom GUD erwähnte Pflegezentren** heraus. (Diejenigen, die nicht im Stadtgebiet liegen sind oben räumlich bereits herausgefiltert worden.)

Annahme vom 1.3.2022: nur jende die mit Gesundheitszentrum beginnen:



In [54]:
#Filter für Pflegezentren?

gdf_poi_pflegezentrum_view_filtered = gdf_poi_pflegezentrum_view_filter_stadtgebiet.query("typen =='Gesundheitszentrum'") 

In [55]:
#gdf_poi_pflegezentrum_view_filtered

,id,adr_inter,adresse,adrzus_int,behindertenparkplatz,bemerkung,ccmail,da,datum,datum_cms,...,zahlungsmittel_internet,zahlungsmittel_schalter,zahlungsmittel_telefon,zvv_label,zvv_link,geometry,typen,index_right,bfs_name,bfsnr
5,poi_pflegezentrum_view.9196,None,Emil-Klöti-Strasse 25,None,None,Betriebsleitung: Andreas Senn,None,Gesundheitszentren für das Alter,202201251158,None,...,NULL;NULL;,NULL;NULL;,NULL;NULL;,Emil-Klöti-Strasse 25,https://www.zvv.ch/zvv/de/fahrplan/fp.html?ton...,POINT (8.51554 47.39907),Gesundheitszentrum,0,Zürich,261.0
6,poi_pflegezentrum_view.9197,None,Helen-Keller-Strasse 12,None,None,Betriebsleitung: Reto Steimen,None,Gesundheitszentren für das Alter,202201251158,None,...,NULL;NULL;,NULL;NULL;,NULL;NULL;,Helen-Keller-Strasse 12,https://www.zvv.ch/zvv/de/fahrplan/fp.html?ton...,POINT (8.59402 47.39991),Gesundheitszentrum,0,Zürich,261.0
7,poi_pflegezentrum_view.9198,None,Witellikerstrasse 19,None,None,Betriebsleitung: Matthias Staub,None,Gesundheitszentren für das Alter,202202041147,None,...,NULL;NULL;,NULL;NULL;,NULL;NULL;,Witellikerstrasse 19,http://online.fahrplan.zvv.ch/bin/query.exe/dn...,POINT (8.57656 47.35242),Gesundheitszentrum,0,Zürich,261.0
11,poi_pflegezentrum_view.9216,None,Kienastenwiesweg 2,None,None,Betriebsleitung: Matthias Staub,None,Gesundheitszentren für das Alter,202202041146,None,...,NULL;NULL;,NULL;NULL;,NULL;NULL;,Kienastenwiesweg 2,http://online.fahrplan.zvv.ch/bin/query.exe/dn...,POINT (8.60180 47.35509),Gesundheitszentrum,0,Zürich,261.0
17,poi_pflegezentrum_view.11182,None,Limmattalstrasse 371,None,None,Betriebsleiter: Urs Leu,None,Gesundheitszentren für das Alter,202201251128,None,...,NULL;NULL;,NULL;NULL;,NULL;NULL;,Limmattalstrasse 371,https://www.zvv.ch/zvv/de/fahrplan/fp.html?ton...,POINT (8.48361 47.40422),Gesundheitszentrum,0,Zürich,261.0
18,poi_pflegezentrum_view.11183,None,Paradiesstrasse 45,None,None,Betriebsleitung: Peter Schuler<BR>,None,Gesundheitszentren für das Alter,202201251128,None,...,NULL;NULL;,NULL;NULL;,NULL;NULL;,Paradiesstrasse 45,https://www.zvv.ch/zvv/de/fahrplan/fp.html?ton...,POINT (8.52680 47.33460),Gesundheitszentrum,0,Zürich,261.0
19,poi_pflegezentrum_view.11211,None,Flurstrasse 130,None,None,Betriebsleitung: Sylvia Stadler Langhart,None,Gesundheitszentren für das Alter,202201251128,None,...,NULL;NULL;,NULL;NULL;,NULL;NULL;,Flurstrasse 130,https://www.zvv.ch/zvv/de/fahrplan/fp.html?ton...,POINT (8.48717 47.37827),Gesundheitszentrum,0,Zürich,261.0
20,poi_pflegezentrum_view.11497,None,Sieberstrasse 22,None,None,Betriebsleitung: Stefan Mosimann,None,Gesundheitszentren für das Alter,202201251128,None,...,NULL;NULL;,NULL;NULL;,NULL;NULL;,Sieberstrasse 22,https://www.zvv.ch/zvv/de/fahrplan/fp.html?ton...,POINT (8.51068 47.36413),Gesundheitszentrum,0,Zürich,261.0


### Step3: Datensätze exportieren

Speichere eine Liste der aktuell gefilterten Alterszentren als CSV lokal ab.

In [56]:
gdf_poi_alterszentrum_view_filtered[['name','adresse', 'plz','ort','geometry']].to_csv("./output/listen/gefilterte_alterszentren_stadt"+date_today+".csv")
gdf_poi_alterszentrum_view_filtered[['name','adresse', 'plz','ort','geometry']]

gdf_poi_pflegezentrum_view_filtered[['id','typen','name','adresse', 'plz','ort','geometry']].to_csv("./output/listen/gefilterte_pflegezentren_stadt"+date_today+".csv")

Wechsle Projektion von WGS84 auf LV95

In [57]:
gdf_poi_alterszentrum_view_filtered_lv95 = gdf_poi_alterszentrum_view_filtered.to_crs(epsg=2056)
#gdf_poi_alterszentrum_view_filtered_lv95['geometry']

gdf_poi_pflegezentrum_view_filtered_lv95 = gdf_poi_pflegezentrum_view_filtered.to_crs(epsg=2056)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [58]:
print(gis_ablagepfad+"export_poi_alterszentrum_view"+date_today+".shp")
print(gis_ablagepfad+"export_poi_pflegezentrum_view"+date_today+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_alterszentrum_view.shp
//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_pflegezentrum_view.shp


In [59]:
gdf_poi_alterszentrum_view_filtered_lv95.to_file(gis_ablagepfad+"export_poi_alterszentrum_view_"+date_today+".shp", encoding='utf-8')

gdf_poi_pflegezentrum_view_filtered_lv95.to_file(gis_ablagepfad+"export_poi_pflegezentrum_view_"+date_today+".shp", encoding='utf-8')
print('done')
#gdf_poi_alterszentrum_view_filtered_lv95.info()

C:\Users\sszsim\AppData\Local\Temp/ipykernel_15716/3801648246.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_poi_alterszentrum_view_filtered_lv95.to_file(gis_ablagepfad+"export_poi_alterszentrum_view_"+date_today+".shp", encoding='utf-8')
C:\Users\sszsim\AppData\Local\Temp/ipykernel_15716/3801648246.py:3: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_poi_pflegezentrum_view_filtered_lv95.to_file(gis_ablagepfad+"export_poi_pflegezentrum_view_"+date_today+".shp", encoding='utf-8')


done


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [60]:
gdf_poi_alterszentrum_view_filtered_lv95.to_file("./output/shapefiles/export_poi_alterszentrum_view"+date_today+".shp", encoding='utf-8')
gdf_poi_alterszentrum_view_filtered_lv95.to_file("./output/geopackages/export_poi_alterszentrum_view"+date_today+".gpkg", driver='GPKG')
#gdf_poi_alterszentrum_view_filtered_lv95.plot()

C:\Users\sszsim\AppData\Local\Temp/ipykernel_15716/4039714076.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_poi_alterszentrum_view_filtered_lv95.to_file("./output/shapefiles/export_poi_alterszentrum_view"+date_today+".shp", encoding='utf-8')


### Alterswohnungen:

Es gibt auch vereinzelt Alterswohnungen ausserhalb der Stadt, daher auch hier noch ein Verschnitt mit der Stadtgrenze.

#### Step 1: Daten vom Geoportal beziehen und prüfen.

In [53]:
url2 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Alterswohnung?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_alterswohnung_view"
gdf_poi_alterswohnungen_view_imp_all = gpd.read_file(url2)
#gdf_poi_alterswohnungen_view_imp_all.info()

#### Step 2: Filtere diejenigen heraus, die nicht benötigt werden

Spatial Join, damit nur jene **innerhalb des Stadtgebiets** vorkommen. Hier sind zwei Möglichkeiten dargestellt, Verschnitt mit Stadtgrenze oder mit den Stadtquartieren. Wichtig ist, dass dabei  auch gerade die Attribute der Geodaten angehängt werden.

In [54]:
gdf_poi_alterswohnungen_view_filter_stadtgebiet = gpd.sjoin(gdf_poi_alterswohnungen_view_imp_all, gdf_adm_stadtgrenze_wgs84, how='inner', op='within')
#gdf_poi_alterswohnungen_view_filter_stadtgebiet.info()

In [55]:
gdf_poi_alterswohnungen_view_filter_stadtgebiet_lv95 = gdf_poi_alterswohnungen_view_filter_stadtgebiet.to_crs(epsg=2056)
#gdf_poi_alterswohnungen_view_filter_stadtgebiet_lv95['geometry']

#### Step3: Datensätze exportieren


Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [56]:
gdf_poi_alterswohnungen_view_filter_stadtgebiet_lv95.to_file(gis_ablagepfad+"export_poi_alterswohnung_view_"+date_today+".shp", encoding='utf-8')

print('done')

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/2967649901.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_poi_alterswohnungen_view_filter_stadtgebiet_lv95.to_file(gis_ablagepfad+"export_poi_alterswohnung_view_"+date_today+".shp", encoding='utf-8')


done


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [57]:
gdf_poi_alterswohnungen_view_filter_stadtgebiet_lv95.to_file("./output/shapefiles/export_poi_alterswohnung_view"+date_today+".shp", encoding='utf-8')
gdf_poi_alterswohnungen_view_filter_stadtgebiet_lv95.to_file("./output/geopackages/export_poi_alterswohnung_view"+date_today+".gpkg", driver='GPKG')
#gdf_poi_alterswohnungen_view_filter_stadtgebiet_lv95.plot()
print('done')

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/2442419897.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_poi_alterswohnungen_view_filter_stadtgebiet_lv95.to_file("./output/shapefiles/export_poi_alterswohnung_view"+date_today+".shp", encoding='utf-8')


done


### Beachvolley (3)

#### Step 1: Daten vom Geoportal beziehen und prüfen.

In [58]:
url3 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_beachvolleyball_view"

typename="poi_beachvolleyball_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

#platziere urlX
imp_file = gpd.read_file(url3)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_beachvolleyball_view_wgs84.head(2)

array(['Schulhaus Stettbach', 'Oberer Letten', 'Schulhaus Im Lee',
       'Schulhaus Buhnrain', 'Freibad Zwischen den Hölzern',
       'Sportanlage Juchof 1', 'Josefwiese', 'Schulhaus Letzi',
       'Sportzentrum Sihlhölzli', 'Schulhaus Lachenzelg',
       'Freibad Seebach', 'Sportzentrum Buchlern', 'Freibad Allenmoos',
       'Sportzentrum Fronwald'], dtype=object)

#### Step2: Datensätze exportieren


In [59]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_beachvolleyball_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [60]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_beachvolleyball_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [61]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_beachvolleyball_view.gpkg


### Bikepark (4)

#### Step 1: Daten vom Geoportal beziehen und prüfen.

- url(X) --> Definiere die URL für den GEOJSON-Service
- typename --> Definiere den Layernamen. Kann an diversen Orten verwendet werden
- imp_name --> Leite den importierten Datensatznamen davon ab und ergänze wgs84
- exp_name --> ohne wgs84. soll heissen er ist per default lv95

In [62]:
url4 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_bikepark_view"

typename="poi_bikepark_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url4)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Bike-Park Allmend', 'Pumptrack Heuried'], dtype=object)

#### Step2: Datensätze exportieren


In [63]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [64]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_bikepark_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [65]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_bikepark_view.gpkg


### Eisbahn (5)

#### Step 1: Daten vom Geoportal beziehen und prüfen.

- url(X) --> Definiere die URL für den GEOJSON-Service
- typename --> Definiere den Layernamen. Kann an diversen Orten verwendet werden
- imp_name --> Leite den importierten Datensatznamen davon ab und ergänze wgs84
- exp_name --> ohne wgs84. soll heissen er ist per default lv95

In [66]:
url5 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_eisbahn_view"

typename="poi_eisbahn_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url5)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Sportzentrum Heuried', 'Kunsteisbahn Oerlikon',
       'Kunsteisbahn Dolder'], dtype=object)

#### Step2: Datensätze exportieren


In [67]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [68]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_eisbahn_view.shp


C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [69]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_eisbahn_view.gpkg


### Flussbad (6)

In [70]:
url6 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_flussbad_view"

typename="poi_flussbad_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url6)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Frauenbad Stadthausquai', 'Männerbad Schanzengraben',
       'Flussbad Unterer Letten', 'Flussbad Oberer Letten',
       'Flussbad Au-Höngg'], dtype=object)

#### Bug 2021-10-19: Doppelter Eintrag Unterer Letten

In [71]:
imp_file_filtered = imp_file.query("objectid != 7257") 
#imp_file_filtered.info()
#imp_file_filtered

#### Step2: Datensätze exportieren


In [72]:
exp_file = imp_file_filtered.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [73]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_flussbad_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [74]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_flussbad_view.gpkg


### Freibad (7)

In [75]:
url7 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_freibad_view"

typename="poi_freibad_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

#place urlX
imp_file = gpd.read_file(url7)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Freibad Letzigraben', 'Freibad Dolder',
       'Freibad Zwischen den Hölzern', 'Freibad Allenmoos',
       'Freibad Auhof', 'Freibad Heuried', 'Freibad Seebach'],
      dtype=object)

#### Step2: Datensätze exportieren


In [76]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [77]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_freibad_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [78]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_freibad_view.gpkg


### Fussballplätze (8)

In [79]:
url8 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_fussballplatz_view"

typename="poi_fussballplatz_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url8)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Rasensportanlage Katzenbach', 'Sportzentrum Hardhof',
       'Rasensportanlage Juchhof 1', 'Sportzentrum Utogrund',
       'Sportzentrum Eichrain', 'Sportzentrum Fronwald',
       'Sportzentrum Heerenschürli', 'Sportzentrum Witikon',
       'Rasensportanlage Neudorf', 'Sportzentrum Heuried',
       'Rasensportanlage Juchhof 2', 'Rasensportanlage Allmend Brunau',
       'Sportzentrum Buchlern', 'Rasensportanlage Forrenweid',
       'Rasenspielfeld Liguster'], dtype=object)

#### Step2: Datensätze exportieren


In [80]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [81]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_fussballplatz_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [82]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_fussballplatz_view.gpkg


### Gemeinschaftszentrum (9)

In [83]:
url9 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Gemeinschaftszentrum?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_gemeinschaftszentrum_view"

typename="poi_gemeinschaftszentrum_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url9)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['GZ Loogarten - Standort Badenerstrasse', 'GZ Höngg',
       'GZ Wollishofen, Standort Neubühl', 'GZ Affoltern',
       'GZ Bachwiesen', 'GZ Wollishofen', 'GZ Oerlikon', 'GZ Heuried',
       'GZ Hirzenbach', 'GZ Hottingen', 'GZ Leimbach', 'GZ Loogarten',
       'GZ Riesbach', 'GZ Seebach', 'GZ Wipkingen', 'GZ Witikon',
       'GZ Leimbach - Standort Manegg',
       'GZ Wollishofen - Standort Albisstrasse',
       'GZ Heuried - Standort Manesse', 'GZ Buchegg', 'GZ Grünau',
       'GZ Höngg - Standort Rütihof', 'GZ Schindlergut',
       'GZ Hottingen - Standort Klosbachstrasse'], dtype=object)

#### Step2: Datensätze exportieren


In [84]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [85]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_gemeinschaftszentrum_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [86]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_gemeinschaftszentrum_view.gpkg


### Hallenbad (10)

In [87]:
url10 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_hallenbad_view"

typename="poi_hallenbad_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url10)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Hallenbad Leimbach', 'Hallenbad Oerlikon', 'Hallenbad Altstetten',
       'Hallenbad Bläsi', 'Hallenbad Bungertwies', 'Hallenbad City',
       'Wärmebad Käferberg'], dtype=object)

#### Step2: Datensätze exportieren


In [88]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [89]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_hallenbad_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [90]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_hallenbad_view.gpkg


### Jugendtreff (11)

In [91]:
url11 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Jugendtreff?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_jugendtreff_view"

typename="poi_jugendtreff_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url11)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['OJA Kreis 3 & 4', 'OJA Affoltern', 'Verein Jugendtreff Kreis 4',
       'OJA Kreis 9 & Hard', 'OJA Wollishofen-Leimbach',
       'OJA Kreis 5/planet 5', 'OJA Kreis 6 & Wipkingen', 'OJA Oerlikon',
       'OJA Schwamendingen',
       'OJA Kreis6&Wipkingen - Standort Bhf.Wipkingen',
       'Jugendkulturhaus Dynamo', 'Verein Jugendarbeit Auzelg (c/o Hort)',
       'OJA Wollishofen&Leimbach Standort Leimbach',
       'Mundwerk Kulturbiotop', 'OJA Kreis 9 & Hard Standort Hardstrasse'],
      dtype=object)

#### Step2: Datensätze exportieren


In [92]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [93]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_jugendtreff_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [94]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_jugendtreff_view.gpkg


### Park (12)

In [95]:
url12 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Park?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_park_view"

typename="poi_park_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url12)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Neumünsteranlage', 'Park Pfingstweid', 'Platzspitz',
       'Ruggächeranlage', 'Arboretum', 'Tiefenbrunnen', 'Althoosstrasse',
       'Im Hagenbrünneli', 'General-Guisan-Quai', 'Glaubten / Müseli',
       'Sonnenberg', 'Savera-Areal', 'Rechberg', 'Lettenareal',
       'Stadthausanlage', 'Bauschänzli', 'ETH Hönggerberg',
       'Holderbachwiese', 'Turbinenplatz', 'Klingenanlage', 'Schärenmoos',
       'GZ Seebach', 'Klopstockwiese', 'Wahlenpark', 'Mattenhof',
       'Felsenrainanlage', 'Schärrerwiese', 'Heiligfeld', 'Riedgraben',
       'Leutschenpark', 'Birchwald', 'Langgrütweg', 'Süssleren',
       'Alter Botanischer Garten', 'Am Katzenbach', 'Patumbah-Park',
       'Villa Tobler', 'Belvoirpark', 'GZ Grünau', 'Blatterwiese',
       'Villa Schönberg', 'Waidberg', 'Wipkingerpark', 'GZ Buchegg',
       'Alfred-Altherr-Terrasse', 'Herzogenmühle-Grünzug', 'Glattwiesen',
       'GZ Hirzenbach', 'Seefeldquai', 'Buchholz', 'Im Gut',
       'Landenberganlage', 'Josefwiese', 'Bäckeranla

#### Step2: Datensätze exportieren


In [96]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [97]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_park_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [98]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_park_view.gpkg


### Picknick (13)

In [99]:
url13 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Erholungs__und_Sporteinrichtungen?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_picknickplatz_view"

typename="poi_picknickplatz_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url13)

imp_file.info()
imp_file['name'].unique()


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 57 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   id                           99 non-null     object  
 1   adr_inter                    0 non-null      object  
 2   adresse                      0 non-null      object  
 3   adrzus_int                   0 non-null      object  
 4   anlageelemente               47 non-null     object  
 5   behindertenparkplatz         0 non-null      object  
 6   bemerkung                    0 non-null      object  
 7   bereich                      0 non-null      object  
 8   ccmail                       0 non-null      object  
 9   da                           99 non-null     object  
 10  datum                        99 non-null     object  
 11  datum_cms                    0 non-null      object  
 12  dep                          99 non-null     object  
 13 

array(['Hohensteinplatz', 'Antenne Fernsehplatz', 'Aegertenweg',
       'Butti-Brunnen', 'Riedenholz Süd', 'Reckenholz', 'Holzhof',
       'Russenbrünneli', 'Heizenholz', 'Kolbenhoferegg', 'Mediker Boden',
       'Schanze Gratstrasse', 'Annaburg', 'Banzwiese',
       'Banzwiesenstr/Hohensteinweg', 'Kamelbuckel', 'Kleeweid',
       'Maneggweg', 'Medikerweg', 'Ruine Manegg', 'Fallätsche',
       'Tscharli Platz', 'Oerliker Wäldli', 'Friesenberg', 'Kolbenhof',
       'Milan', 'Wasserrad', 'Spinne', 'Hasenrainhütte',
       'Hurdäckerstrasse', 'Höckler', 'Hochwacht', 'ARA  Adliswil',
       'Buchenweg', 'Burgwies', 'Mädikon', 'Chalberweid', 'Gabel',
       'Tamboureneiche', 'Entlisbergkopf', 'Rüteliweg', 'Bergholz',
       'Kappeli', 'Schachtisch', 'Tobelholzweg', 'Rumpelhalde',
       'Rötelterrasse', 'Unterer Spielplatz', 'Eichhörnliweg',
       'Oberer Spielplatz', 'Skatmatt', 'Rondell', 'Reservoir Käferberg',
       'Käferberg-Weiher', 'Waidbergweg', 'Zweifelseiche', 'Elefant',
       

#### Step2: Datensätze exportieren


In [100]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [101]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_picknickplatz_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [102]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_picknickplatz_view.gpkg


### Quartiertreffhaus (14)

In [103]:
url14 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Quartiertreff?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_quartiertreffhaus_view"

typename="poi_quartiertreffhaus_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url14)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Quartierraum Fritschiwiese', 'Quartiertreff Fluntern',
       'Quartiertreff Hirslanden', 'Quartiertreff Zehntenhaus',
       'Quartierhaus Schütze (Mehrzweckraum)',
       'Quartiertreff Altstadthaus', 'Schopf - Verein Hochneun',
       'Quartiertreff Waldgarten', 'Quartierzentrum Schütze',
       'Quartiertreff Hüttenkopf', 'Quartierbaracke Auzelg', 'Stadtgeiss',
       'Zentrum Karl der Grosse', 'Quartierraum Viadukt',
       'Quartierraum Zentralstrasse', 'Quartierräume Bullinger',
       'Kulturbahnhof Affoltern, KuBaA', 'Kiosk Josefwiese',
       'Quartierräume Hardau', 'Solino', 'Quartierhof Wynegg',
       'Quartierhaus Kreis 6', 'Quartiertreff Enge', 'Spielbaracke',
       'Quartierzentrum Bäckeranlage'], dtype=object)

#### Step2: Datensätze exportieren


In [104]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [105]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_quartiertreffhaus_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [106]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_quartiertreffhaus_view.gpkg


### Seebad (15)

In [107]:
url15 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_seebad_view"

typename="poi_seebad_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url15)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Seebad Enge', 'Seebad Katzensee', 'Seebad Utoquai',
       'Strandbad Mythenquai', 'Strandbad Tiefenbrunnen',
       'Strandbad Wollishofen'], dtype=object)

#### Step2: Datensätze exportieren


In [108]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [109]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_seebad_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [110]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_seebad_view.gpkg


### Skateranlage (16)

In [111]:
url16 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_skateranlage_view"

typename="poi_skateranlage_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url16)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Freestyleanlage Leimbach', 'Freestyle-Halle Werdhölzli',
       'Freestyleanlage Buchlern', 'Freestyleanlage Heerenschürli',
       'Freestyleanlage Hohlstrasse', 'Freestyleanlage Fronwald',
       'Freestyleanlage Kornhausbrücke', 'Freestyle-Park Allmend'],
      dtype=object)

#### Step2: Datensätze exportieren


In [112]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [113]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_skateranlage_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [114]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_skateranlage_view.gpkg


### Spielplatz (17)

In [115]:
url17 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Spielgeraetekataster?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_spielplatz_view"

typename="poi_spielplatz_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url17)

#imp_file.info()
imp_file['name'].unique()
#imp_file.head()


array(['Manessestrasse / Weststrasse',
       'Rote Fabrik                                                           ',
       'Langgrüt - Altweg                                                     ',
       'Lutherwiese                                                           ',
       'Süsslerenanlage                                                       ',
       'Kanzleiwiese                                                          ',
       'Kasernenareal                                                         ',
       'Pflanzschulwiese                                                      ',
       'Zurlindenhof II                                                       ',
       'Irringersteig                                                         ',
       'Rebenweg                                                              ',
       'Steinstrasse / Gotthelfstrasse', 'Weststrasse / Werdstrasse',
       'Steinstrasse / Erlachstrasse', 'Manesseplatz',
       'Eibenstrasse / Man

#### Step2: Datensätze exportieren


In [116]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [117]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_spielplatz_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [118]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_spielplatz_view.gpkg


### Sporthalle (18)

In [119]:
url18 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_sporthalle_view"

typename="poi_sporthalle_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url18)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Sportzentrum Fronwald', 'Dreifachhalle Im Birch',
       'Spezialhalle Leimbach', 'Dreifachhalle Hofacker',
       'Schul- und Sportanlage Döltschi',
       'Schul- und Sportanlage Schauenberg',
       'Schul- und Sportanlage Apfelbaum',
       'Schul- und Sporthalle Hirzenbach', 'Dreifachhalle Hardau',
       'Sportzentrum Sihlhölzli', 'Schul- und Sportanlage Leutschenbach',
       'Saalsporthalle', 'Dreifachhalle Blumenfeld',
       'Schul- und Sportanlage Falletsche', 'Sportzentrum Utogrund',
       'Schul- und Sportanlage Buchwiesen',
       'Schul- und Sportanlage Stettbach', 'Spezialhalle Riesbach',
       'Sportzentrum Josef', 'ASVZ Rämibühl', 'Sport Center Hönggerberg'],
      dtype=object)

#### Step2: Datensätze exportieren


In [120]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [121]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_sporthalle_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [122]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_sporthalle_view.gpkg


### Stadion (19)

In [123]:
url19 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_stadion_view"

typename="poi_stadion_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url19)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Stadion Letzigrund'], dtype=object)

#### Step2: Datensätze exportieren


In [124]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [125]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_stadion_view.shp


C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [126]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_stadion_view.gpkg


### Tennisplatz (20)

In [127]:
url20 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_tennisplatz_view"

typename="poi_tennisplatz_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url20)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Tennisanlage Hardhof', 'Tennisanlage Buchlern',
       'Tennisanlage Eichrain', 'Tennisanlage Fronwald',
       'Tennisanlage Lengg', 'Tennisanlage Mythenquai',
       'Tennisanlage Frauental', 'Tennisanlage Ueberlandstrasse'],
      dtype=object)

#### Step2: Datensätze exportieren


In [128]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [129]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_tennisplatz_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [130]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_tennisplatz_view.gpkg


### Waldhütte (21)

In [131]:
url21 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Erholungs__und_Sporteinrichtungen?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_waldhuette_view"

typename="poi_waldhuette_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url21)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Waldhütte Escherhöhe', 'Waldhüsli', 'Moosholz', 'Huebbach',
       'Hasenrain', 'Forsthaus Frauenmatt', 'Loorenkopf, Adlisberg',
       'Adlisberghütte', 'Köhlerstrasse', 'Buschberg',
       'Hohensteinstrasse', 'Annaburg', 'Elefant', 'Stöckentobelhütte',
       'Tscharli Platz', 'Waldhütte Ziegelhöhe'], dtype=object)

#### Step2: Datensätze exportieren


In [132]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile

In [133]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_waldhuette_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [134]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_waldhuette_view.gpkg


### Zürifit (22)

In [135]:
url22 = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=poi_zuerifit_view"

typename="poi_zuerifit_view"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url22)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Zürifit Neunbrunnenstrasse', 'Zürifit Irchelpark',
       'Zürifit Hardhof', 'Zürifit Brunau'], dtype=object)

#### Step2: Datensätze exportieren


In [136]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [137]:
exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1368340669.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file(gis_ablagepfad+"export_"+typename+date_today+".shp", encoding='utf-8')


//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_poi_zuerifit_view.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [138]:
exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackages/export_"+typename+date_today+".gpkg", driver='GPKG')
#exp_file.plot()
print("GPKG exportiert nach: "+"./output/geopackages/export_"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/1312096258.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/export_"+typename+date_today+".shp", encoding='utf-8')


GPKG exportiert nach: ./output/geopackages/export_poi_zuerifit_view.gpkg


### ----------------------------------------
### Vorlage (X)

#### Step 1: Daten vom Geoportal beziehen und prüfen.

- url(X) --> Definiere die URL für den GEOJSON-Service
- typename --> Definiere den Layernamen. Kann an diversen Orten verwendet werden
- imp_name --> Leite den importierten Datensatznamen davon ab und ergänze wgs84
- exp_name --> ohne wgs84. soll heissen er ist per default lv95

In [139]:
urlX = "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Sport?service=WFS&version=1.1.0&request=GetFeature&outputFormat=GeoJSON&typename=X"

typename="X"

imp_file="gdf_"+typename+"_wgs84"
exp_file="gdf_"+typename

imp_file = gpd.read_file(url4)

#imp_file.info()
imp_file['name'].unique()
#gdf_poi_bikepark_view_wgs84.head()

array(['Bike-Park Allmend', 'Pumptrack Heuried'], dtype=object)

#### Step2: Datensätze exportieren


In [140]:
exp_file = imp_file.to_crs(epsg=2056)
#gdf_poi_bikepark_view['geometry'].head(1)

Speichere den Geodatensatz für Stefanie auf `O:\GIS\Daten\Vektor\2021\quartierspiegel` als Shapefile


In [141]:
#exp_file.to_file(gis_ablagepfad+"export_"+typename+".shp", encoding='utf-8')
print(gis_ablagepfad+"export_"+typename+".shp")

//szh/ssz/data/GIS/Daten/Vektor/2022/quartierspiegel/export_X.shp


Speichere sie auch im Arbeitsverzeichnis als Shapefile und Geopackage ab

In [142]:
exp_file.to_file("./output/shapefiles/"+typename+date_today+".shp", encoding='utf-8')
exp_file.to_file("./output/geopackage/"+typename+date_today+".gpkg", driver='GPKG')
#gdf_poi_bikepark_view.plot()
print("GPKG exportiert nach: "+"./output/geopackage/"+typename+date_today+".gpkg")

C:\Users\sszsim\AppData\Local\Temp/ipykernel_4072/3805502502.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  exp_file.to_file("./output/shapefiles/"+typename+date_today+".shp", encoding='utf-8')


DriverError: sqlite3_open(./output/geopackage/X.gpkg) failed: unable to open database file